# 02 · Data Cleaning

**Goal:** Fix anomalies and outliers in the application data, flag heavily-missing columns, encode categorical variables and save the cleaned dataset.

| | |
|---|---|
| **Input** | `data/raw/application_train.csv` |
| **Output** | `data/processed/application_train_cleaned.csv` |
| **Next** | `03_feature_engineering.ipynb` |

## 1. Load the data

In [ ]:
import pandas as pd
import numpy as np

app_train = pd.read_csv('../data/raw/application_train.csv')
print("Shape:", app_train.shape)

Shape: (307511, 122)


## 2. Fix the `DAYS_EMPLOYED` placeholder
`365243` is a placeholder, not a real value. Flag those rows first so the information is kept, then replace the placeholder with `NaN`.

In [ ]:
# Fix DAYS_EMPLOYED anomaly
# Flag anomalous rows BEFORE fixing the values
app_train['DAYS_EMPLOYED_ANOMALY'] = (app_train['DAYS_EMPLOYED'] == 365243).astype(int)
print("Anomalous rows:", app_train['DAYS_EMPLOYED_ANOMALY'].sum())

# Replace placeholder with NaN
app_train['DAYS_EMPLOYED'] = app_train['DAYS_EMPLOYED'].replace(365243, np.nan)

app_train['DAYS_EMPLOYED'].describe()

Anomalous rows: 55374


count    252137.000000
mean      -2384.169325
std        2338.360162
min      -17912.000000
25%       -3175.000000
50%       -1648.000000
75%        -767.000000
max           0.000000
Name: DAYS_EMPLOYED, dtype: float64

## 3. Convert age to years
`DAYS_BIRTH` is a negative day count, so convert it to a positive age in years.

In [ ]:
# Convert DAYS_BIRTH to age in years
app_train['AGE_YEARS'] = -app_train['DAYS_BIRTH'] / 365
app_train['AGE_YEARS'].describe()

count    307511.000000
mean         43.936973
std          11.956133
min          20.517808
25%          34.008219
50%          43.150685
75%          53.923288
max          69.120548
Name: AGE_YEARS, dtype: float64

## 4. Income outliers
Inspect the largest incomes, then cap income at the 99.5th percentile to limit the influence of extreme values.

In [ ]:
# Investigate the income outlier
app_train.nlargest(5, 'AMT_INCOME_TOTAL')[['SK_ID_CURR', 'AMT_INCOME_TOTAL', 'TARGET', 'NAME_INCOME_TYPE', 'OCCUPATION_TYPE']]

,SK_ID_CURR,AMT_INCOME_TOTAL,TARGET,NAME_INCOME_TYPE,OCCUPATION_TYPE
12840,114967,117000000.0,1,Working,Laborers
203693,336147,18000090.0,0,Commercial associate,NaN
246858,385674,13500000.0,0,Commercial associate,NaN
77768,190160,9000000.0,0,Working,Managers
131127,252084,6750000.0,0,Working,Laborers


### Cap extreme incomes

In [ ]:
# Cap income at the 99.5th percentile to reduce influence of extreme outliers
income_cap = app_train['AMT_INCOME_TOTAL'].quantile(0.995)
print("Cap value:", income_cap)

app_train['AMT_INCOME_TOTAL_CAPPED'] = app_train['AMT_INCOME_TOTAL'].clip(upper=income_cap)
app_train['AMT_INCOME_TOTAL_CAPPED'].describe()

Cap value: 630000.0


count    307511.000000
mean     167136.388797
std       87919.401895
min       25650.000000
25%      112500.000000
50%      147150.000000
75%      202500.000000
max      630000.000000
Name: AMT_INCOME_TOTAL_CAPPED, dtype: float64

## 5. Missing-value flags
For columns with more than 40% missing values, add a binary `_MISSING` flag so the model can use missingness as a signal.

In [ ]:
# Missing-value flags for high-missingness columns
missing_pct = app_train.isnull().mean() * 100
high_missing_cols = missing_pct[missing_pct > 40].index.tolist()

# Build all flag columns at once, then concat in a single operation
missing_flags = app_train[high_missing_cols].isnull().astype(int)
missing_flags.columns = [f'{col}_MISSING' for col in high_missing_cols]

app_train = pd.concat([app_train, missing_flags], axis=1)

print(f"Created missing-flags for {len(high_missing_cols)} columns")

Created missing-flags for 49 columns


## 6. Anomalous gender values
Check `CODE_GENDER` for invalid categories, then drop the rows with `XNA`.

In [ ]:
# Check CODE_GENDER
app_train['CODE_GENDER'].value_counts()

CODE_GENDER
F      202448
M      105059
XNA         4
Name: count, dtype: int64

In [ ]:
# Drop anomalous gender values
app_train = app_train[app_train['CODE_GENDER'] != 'XNA']
print("New shape:", app_train.shape)

New shape: (307507, 174)


## 7. Encode categorical columns
List the text columns and their cardinality, then one-hot encode them (with a column for missing values).

In [ ]:
# Encoding the categorical columns
categorical_cols = app_train.select_dtypes(include=['object', 'string']).columns.tolist()
print(f"Number of categorical columns: {len(categorical_cols)}")
print(categorical_cols)

for col in categorical_cols:
    print(f"\n{col}: {app_train[col].nunique()} unique values")

Number of categorical columns: 16
['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE']

NAME_CONTRACT_TYPE: 2 unique values

CODE_GENDER: 2 unique values

FLAG_OWN_CAR: 2 unique values

FLAG_OWN_REALTY: 2 unique values

NAME_TYPE_SUITE: 7 unique values

NAME_INCOME_TYPE: 8 unique values

NAME_EDUCATION_TYPE: 5 unique values

NAME_FAMILY_STATUS: 6 unique values

NAME_HOUSING_TYPE: 6 unique values

OCCUPATION_TYPE: 18 unique values

WEEKDAY_APPR_PROCESS_START: 7 unique values

ORGANIZATION_TYPE: 58 unique values

FONDKAPREMONT_MODE: 4 unique values

HOUSETYPE_MODE: 3 unique values

WALLSMATERIAL_MODE: 7 unique values

EMERGENCYSTATE_MODE: 2 unique values


### One-hot encode

In [ ]:
# One-hot encode the categorical columns
app_train_encoded = pd.get_dummies(app_train, columns=categorical_cols, dummy_na=True)
print("Shape before encoding:", app_train.shape)
print("Shape after encoding:", app_train_encoded.shape)

Shape before encoding: (307507, 174)
Shape after encoding: (307507, 313)


## 8. Save the cleaned dataset

In [ ]:
app_train_encoded.to_csv('../data/processed/application_train_cleaned.csv', index=False)
print("Saved cleaned dataset:", app_train_encoded.shape)

Saved cleaned dataset: (307507, 313)


## 9. Validation checks
Confirm there are no duplicate columns, the target distribution is intact, and the saved file reloads with the expected shape.

In [ ]:
# Confirm no duplicate columns
print("Duplicate columns:", app_train_encoded.columns[app_train_encoded.columns.duplicated()].tolist())

# Confirm TARGET is still intact and correctly distributed
print(app_train_encoded['TARGET'].value_counts(normalize=True) * 100)

# Confirm file saved correctly by reloading it
check = pd.read_csv('../data/processed/application_train_cleaned.csv')
print("Reloaded shape:", check.shape)

Duplicate columns: []
TARGET
0    91.927013
1     8.072987
Name: proportion, dtype: float64
Reloaded shape: (307507, 313)
